# Flexible Search via GPU

This notebook searches over $\prod_i [m_i,n_i]$ using the improved dimension computation algorithm loaded from the same external file as before.


Candidate generation, initial filtering, and dynamic queue pruning are now vectorized and use fused CuPy CUDA kernels when available. 

Custom bounds, CSV checkpointing, and safe interruption are preserved.


In [1]:
# %load dim_backprop_gpu_only.py
"""Exact GPU based computation of polynomial-network neurovariety dimensions. This module has no SageMath or NumPy backend.   

It requires CuPy and a CUDA-capable NVIDIA GPU. CuPy replaces Numpy and provides GPU-accelerated array operations.

Comment: I used an ASUS Dual GeForce RTX 4070 Super.

The public function keeps the original calling convention of the Sage implementation: compute_dimension(network_widths, network_exponent) so that I can use the previous notebooks.

The returned tuple like before is: (sizes, exponent, ambient_dim, expected_dim, dimension, defect).

The computation is exact over the finite fields listed in ``DEFAULT_PRIMES``. One might want to pick larger primes for large runs.

All samples and all output-coordinate pullbacks are differentiated in one batched GPU computation.  

The final Jacobian rank is computed modulo each prime by GPU-parallel row elimination.
"""

from __future__ import annotations

from math import comb
from typing import Iterator, Sequence

try:
    import cupy as cp
except ImportError as exc:
    raise RuntimeError(
        "This GPU-only module requires CuPy. Install the CuPy package that "
        "matches your CUDA version, for example `pip install cupy-cuda12x`."
    ) from exc


DEFAULT_PRIMES = (100003, 100153)
_INT64_MAX = (1 << 63) - 1


def _require_cuda() -> None:
    """Raise a clear error unless a usable CUDA device is visible."""
    try:
        device_count = int(cp.cuda.runtime.getDeviceCount())
    except cp.cuda.runtime.CUDARuntimeError as exc:
        raise RuntimeError(
            "CuPy is installed, but CUDA could not be initialized. Check the "
            "NVIDIA driver, CUDA/CuPy compatibility, and notebook kernel."
        ) from exc

    if device_count < 1:
        raise RuntimeError("No CUDA-capable GPU is visible to CuPy.")


def gpu_information() -> dict[str, object]:
    """Return basic information about the CUDA device used by this module."""
    _require_cuda()
    device_id = int(cp.cuda.runtime.getDevice())
    properties = cp.cuda.runtime.getDeviceProperties(device_id)
    name = properties["name"]
    if isinstance(name, bytes):
        name = name.decode("utf-8", errors="replace")
    return {
        "device_id": device_id,
        "name": name,
        "device_count": int(cp.cuda.runtime.getDeviceCount()),
        "cupy_version": cp.__version__,
        "cuda_runtime_version": int(cp.cuda.runtime.runtimeGetVersion()),
        "driver_version": int(cp.cuda.runtime.driverGetVersion()),
    }


def _is_prime(value: int) -> bool:
    """Deterministic Miller--Rabin test for unsigned 64-bit integers to check for primality."""
    if value < 2:
        return False

    small_primes = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)
    if value in small_primes:
        return True
    if any(value % prime == 0 for prime in small_primes):
        return False

    odd_part = value - 1
    power_of_two = 0
    while odd_part % 2 == 0:
        power_of_two += 1
        odd_part //= 2

    # Deterministic for every n < 2^64.
    for base in (2, 325, 9375, 28178, 450775, 9780504, 1795265022):
        if base % value == 0:
            continue
        witness = pow(base, odd_part, value)
        if witness in (1, value - 1):
            continue
        for _ in range(power_of_two - 1):
            witness = (witness * witness) % value
            if witness == value - 1:
                break
        else:
            return False

    return True


def _weak_compositions(total: int, length: int) -> Iterator[tuple[int, ...]]:
    """Yield nonnegative ``length``-tuples whose entries sum to ``total``. 
    These will be used to form a unisolvent evaluation set for 
    homogeneous polynomials of total degree ``total`` in 
    ``length``-many variables. Tuples should be generated in lexicographic order."""
    if length == 0:
        if total == 0:
            yield ()
        return
    if length == 1:
        yield (total,)
        return

    for first in range(total + 1):
        for rest in _weak_compositions(total - first, length - 1):
            yield (first,) + rest


def _unisolvent_samples(input_dim: int, degree: int, prime: int):
    r"""Construct the homogeneous interpolation points directly on the GPU.

    On the chart ``x_0 = 1``, homogeneous degree-``degree`` forms become
    polynomials of total degree at most ``degree`` in ``input_dim - 1``
    variables.  The integer simplex is an unisolvent evaluation set when
    ``prime > degree``.
    """
    if input_dim < 1:
        raise ValueError("the input layer must have positive width")
    if degree < 0:
        raise ValueError("degree must be nonnegative")
    if prime <= degree:
        raise ValueError(
            f"prime {prime} must exceed polynomial degree {degree}"
        )

    expected = comb(degree + input_dim - 1, input_dim - 1)
    if input_dim == 1:
        return cp.ones((1, 1), dtype=cp.int64)

    points: list[tuple[int, ...]] = []
    for total in range(degree + 1):
        for alpha in _weak_compositions(total, input_dim - 1):
            points.append((1,) + alpha)

    if len(points) != expected:
        raise RuntimeError(
            f"internal sample-count error: got {len(points)}, expected {expected}"
        )

    return cp.asarray(points, dtype=cp.int64) % prime


def _mod_pow(base, exponent: int, prime: int):
    """Elementwise matrix + mod p exponentiation on the GPU."""
    if exponent < 0:
        raise ValueError("exponent must be nonnegative")

    result = cp.ones_like(base, dtype=cp.int64)
    if exponent == 0:
        return result

    power = base.astype(cp.int64, copy=False) % prime
    remaining = int(exponent)
    while remaining:
        if remaining & 1:
            result = (result * power) % prime
        remaining >>= 1
        if remaining:
            power = (power * power) % prime

    return result


def _check_dot_product_safety(widths: Sequence[int], prime: int) -> None:
    """Prevent signed int64 overflow before a matrix product is reduced."""
    largest_inner_dimension = max(int(width) for width in widths)
    worst_case = largest_inner_dimension * (prime - 1) ** 2
    if worst_case > _INT64_MAX:
        raise OverflowError(
            "An int64 GPU matrix product may overflow before reduction modulo "
            "the prime. Use a smaller prime or narrower layers."
        )


def _random_weights(
    widths: Sequence[int], prime: int, seed: int
) -> list[cp.ndarray]:
    """Generate all network weight matrices directly in GPU memory.
    The random number generator is seeded for reproducibility. 
    The generation is uniform over ``0, 1, ..., prime - 1``.
    """
    rng = cp.random.RandomState(seed)
    return [
        rng.randint(
            0,
            prime,
            size=(out_width, in_width),
            dtype=cp.int64,
        )
        for in_width, out_width in zip(widths[:-1], widths[1:])
    ]


def _parameter_offsets(
    weights: Sequence[cp.ndarray],
) -> tuple[list[int], int]:
    offsets: list[int] = []
    total = 0
    for weight in weights:
        offsets.append(total)
        total += int(weight.shape[0] * weight.shape[1])
    return offsets, total


def _batched_weight_jacobian(
    weights: Sequence[cp.ndarray],
    samples: cp.ndarray,
    exponent: int,
    prime: int,
) -> cp.ndarray:
    """Evaluate every output/weight derivative at every sample on the GPU.

    The result has shape ``(number_of_samples, output_width, num_parameters)``.
    Weight matrices are flattened layer by layer in row-major order, matching
    the ordering used by Sage's matrix ``list()`` method in the old code.
    """
    if exponent < 1:
        raise ValueError("network exponent must be at least 1")
    if not weights:
        raise ValueError("the network must contain at least one weight layer")

    activations = [samples]
    preactivations: list[cp.ndarray] = []
    activation = samples

    # Hidden layers use z -> z^exponent; the final layer is linear.
    for weight in weights[:-1]:
        preactivation = cp.matmul(activation, weight.T) % prime
        preactivations.append(preactivation)
        activation = _mod_pow(preactivation, exponent, prime)
        activations.append(activation)

    batch_size = int(samples.shape[0])
    output_width = int(weights[-1].shape[0])
    offsets, num_parameters = _parameter_offsets(weights)

    jacobian = cp.zeros(
        (batch_size, output_width, num_parameters), dtype=cp.int64
    )

    # Final linear layer.
    final_input = activations[-1]
    final_offset = offsets[-1]
    final_input_width = int(weights[-1].shape[1])
    for output_index in range(output_width):
        start = final_offset + output_index * final_input_width
        stop = start + final_input_width
        jacobian[:, output_index, start:stop] = final_input

    if len(weights) == 1:
        return jacobian

    # Derivatives of all output coordinates with respect to the last hidden
    # preactivation: shape (sample, output, hidden neuron).
    delta = cp.broadcast_to(
        weights[-1][None, :, :],
        (batch_size, output_width, int(weights[-1].shape[1])),
    ).copy()
    derivative = (
        (exponent % prime)
        * _mod_pow(preactivations[-1], exponent - 1, prime)
    ) % prime
    delta = (delta * derivative[:, None, :]) % prime

    # Hidden layers from last to first.
    for layer_index in range(len(weights) - 2, -1, -1):
        weight = weights[layer_index]
        layer_input = activations[layer_index]

        gradient = (
            delta[:, :, :, None] * layer_input[:, None, None, :]
        ) % prime

        start = offsets[layer_index]
        stop = start + int(weight.shape[0] * weight.shape[1])
        jacobian[:, :, start:stop] = gradient.reshape(
            batch_size, output_width, stop - start
        )

        if layer_index > 0:
            delta = cp.matmul(delta, weight) % prime
            derivative = (
                (exponent % prime)
                * _mod_pow(
                    preactivations[layer_index - 1], exponent - 1, prime
                )
            ) % prime
            delta = (delta * derivative[:, None, :]) % prime

    return jacobian


def _rank_mod_prime_gpu(
    matrix: cp.ndarray,
    prime: int,
    workspace_bytes: int = 512 * 1024**2,
) -> int:
    """Compute exact matrix rank over GF(prime) using GPU row operations.
    
    Question for future: Can we replace this with a predefined CuPy function?
    """
    if matrix.ndim != 2:
        raise ValueError("rank input must be a matrix")
    if workspace_bytes <= 0:
        raise ValueError("workspace_bytes must be positive")

    reduced = matrix.astype(cp.int64, copy=True) % prime

    # Eliminate along the smaller dimension.
    if reduced.shape[1] > reduced.shape[0]:
        reduced = reduced.T.copy()

    nrows, ncols = map(int, reduced.shape)
    pivot_row = 0

    for column in range(ncols):
        if pivot_row == nrows:
            break

        nonzero = reduced[pivot_row:, column] != 0
        if not bool(cp.any(nonzero).item()):
            continue

        pivot = pivot_row + int(cp.argmax(nonzero).item())
        if pivot != pivot_row:
            temporary = reduced[pivot_row, :].copy()
            reduced[pivot_row, :] = reduced[pivot, :]
            reduced[pivot, :] = temporary

        pivot_value = int(reduced[pivot_row, column].item())
        inverse = pow(pivot_value, prime - 2, prime)
        reduced[pivot_row, column:] = (
            reduced[pivot_row, column:] * inverse
        ) % prime

        # The row updates are parallel CUDA kernels. Chunking bounds temporary
        # memory usage for large Jacobians.
        remaining_columns = ncols - column
        bytes_per_row = max(1, 3 * remaining_columns * 8)
        rows_per_chunk = max(1, workspace_bytes // bytes_per_row)

        start = pivot_row + 1
        while start < nrows:
            stop = min(nrows, start + rows_per_chunk)
            factors = reduced[start:stop, column].copy()
            reduced[start:stop, column:] = (
                reduced[start:stop, column:]
                - factors[:, None] * reduced[pivot_row, column:][None, :]
            ) % prime
            start = stop

        pivot_row += 1

    return pivot_row


def compute_dimension(
    network_widths: Sequence[int],
    network_exponent: int,
    *,
    primes: Sequence[int] = DEFAULT_PRIMES,
    seed: int = 20260630,
    rank_workspace_bytes: int = 512 * 1024**2,
    verbose: bool = False,
):
    """Compute the neurovariety dimension entirely with the CUDA backend.

    Parameters
    ----------
    network_widths:
        Layer widths ``[d0, d1, ..., dL]``.
    network_exponent:
        Common hidden-layer activation exponent.
    primes:
        Prime moduli used to cross-check the generic rank.
    seed:
        Base random seed for the network weights.
    rank_workspace_bytes:
        Approximate upper bound for temporary elimination workspace.
    verbose:
        Print GPU and rank information.

    Returns
    -------
    tuple
        ``(sizes, exponent, ambient_dim, expected_dim, dimension, defect)``.
    """
    _require_cuda()

    widths = tuple(int(width) for width in network_widths)
    exponent = int(network_exponent)

    if len(widths) < 2:
        raise ValueError("network_widths must contain input and output widths")
    if any(width <= 0 for width in widths):
        raise ValueError("all network widths must be positive")
    if exponent < 1:
        raise ValueError("network_exponent must be at least 1")
    if not primes:
        raise ValueError("at least one prime is required")

    degree = exponent ** (len(widths) - 2)
    ambient_per_output = comb(degree + widths[0] - 1, widths[0] - 1)
    ambient_dim = ambient_per_output * widths[-1]
    num_parameters = sum(
        in_width * out_width
        for in_width, out_width in zip(widths[:-1], widths[1:])
    )

    dimensions: list[int] = []

    if verbose:
        info = gpu_information()
        print(
            f"GPU {info['device_id']}: {info['name']} | "
            f"CuPy {info['cupy_version']} | "
            f"CUDA runtime {info['cuda_runtime_version']}"
        )

    for prime_index, prime_value in enumerate(primes):
        prime = int(prime_value)
        if not _is_prime(prime):
            raise ValueError(f"modulus {prime} is not prime")
        _check_dot_product_safety(widths, prime)

        samples = _unisolvent_samples(widths[0], degree, prime)
        weights = _random_weights(
            widths,
            prime,
            seed + 1_000_003 * prime_index + prime,
        )

        jacobian = _batched_weight_jacobian(
            weights,
            samples,
            exponent,
            prime,
        )
        rank_matrix = jacobian.reshape(
            ambient_per_output * widths[-1], num_parameters
        )
        dimension = _rank_mod_prime_gpu(
            rank_matrix,
            prime,
            workspace_bytes=rank_workspace_bytes,
        )
        dimensions.append(dimension)

        # Ensure kernels for this prime have completed before reporting and
        # releasing memory.
        cp.cuda.Stream.null.synchronize()

        if verbose:
            print(
                f"prime={prime}, samples={ambient_per_output}, "
                f"rank_matrix={tuple(rank_matrix.shape)}, rank={dimension}"
            )

        del rank_matrix, jacobian, weights, samples
        cp.get_default_memory_pool().free_all_blocks()

    if not all(dimension == dimensions[0] for dimension in dimensions):
        raise ValueError(
            "different dimensions over finite fields: " + str(dimensions)
        )

    naive_bound = sum(
        (in_width - 1) * out_width
        for in_width, out_width in zip(widths[:-1], widths[1:])
    ) + widths[-1]
    expected_dim = min(ambient_dim, naive_bound)
    dimension = dimensions[0]

    return (
        list(widths),
        exponent,
        ambient_dim,
        expected_dim,
        dimension,
        expected_dim - dimension,
    )


c:\Users\daoke\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


# Search Algorithm

In [2]:
import os
import ast
import math
import random as py_random
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import cupy as cp
    _CUPY_AVAILABLE = True
except Exception:
    cp = None
    _CUPY_AVAILABLE = False


c:\Users\daoke\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Helper Functions


def calculate_parameter_count(hidden: tuple, d_0: int, d_h: int) -> int:
    """Preserve the notebook's original parameter-count bound exactly."""
    sizes = [d_0] + list(hidden) + [d_h]
    return sum(m * n for m, n in zip(sizes[:-1], sizes[1:]))


def is_less_or_equal(t1: tuple, t2: tuple) -> bool:
    """Return True when t1 is coordinatewise less than or equal to t2."""
    return len(t1) == len(t2) and all(a <= b for a, b in zip(t1, t2))


def _as_bool(value: object) -> bool:
    """Parse booleans that may have been round-tripped through a CSV."""
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes"}
    return bool(value)


def _parse_architecture(value: object) -> tuple[int, ...]:
    if isinstance(value, str):
        value = ast.literal_eval(value)
    return tuple(int(x) for x in value)


def _insert_minimal(
    antichain: set[tuple[int, ...]],
    point: tuple[int, ...],
) -> set[tuple[int, ...]]:
    """Insert a point into an antichain of coordinatewise minimal points."""
    if any(is_less_or_equal(other, point) for other in antichain):
        return antichain
    return {
        other for other in antichain
        if not is_less_or_equal(point, other)
    } | {point}


def _insert_maximal(
    antichain: set[tuple[int, ...]],
    point: tuple[int, ...],
) -> set[tuple[int, ...]]:
    """Insert a point into an antichain of coordinatewise maximal points."""
    if any(is_less_or_equal(point, other) for other in antichain):
        return antichain
    return {
        other for other in antichain
        if not is_less_or_equal(other, point)
    } | {point}


def _minimal_antichain(
    points: Iterable[tuple[int, ...]],
) -> set[tuple[int, ...]]:
    result: set[tuple[int, ...]] = set()
    for point in sorted(set(points), key=lambda x: (sum(x), x)):
        result = _insert_minimal(result, point)
    return result


def _maximal_antichain(
    points: Iterable[tuple[int, ...]],
) -> set[tuple[int, ...]]:
    result: set[tuple[int, ...]] = set()
    for point in sorted(set(points), key=lambda x: (-sum(x), x)):
        result = _insert_maximal(result, point)
    return result


def _save_csv(df: pd.DataFrame, csv_path: Path) -> None:
    """Save to the same requested CSV path; no companion state file is used."""
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)


def _append_result(
    df: pd.DataFrame,
    result: dict,
    *,
    hidden: tuple[int, ...],
    h: int,
    exponent: int,
    d_0: int,
    d_h: int,
    ambient_dim: int,
) -> pd.DataFrame:
    """Append one evaluator result while preserving the original CSV schema."""
    row = dict(result)
    row.setdefault("h", h)
    row.setdefault("exponent", exponent)
    row.setdefault("architecture", str([d_0, *hidden, d_h]))
    row.setdefault("num_parameters", calculate_parameter_count(hidden, d_0, d_h))
    row.setdefault("ambient_dimension", ambient_dim)
    row.setdefault("is_minimal", False)

    architecture = _parse_architecture(row["architecture"])
    row["architecture"] = str(list(architecture))

    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return df.drop_duplicates(
        subset=["h", "exponent", "architecture"],
        keep="last",
    )


def _recompute_minimality(df: pd.DataFrame) -> pd.DataFrame:
    """Recompute minimality relative to filling architectures found so far."""
    if df.empty:
        return df

    df = df.copy()
    df["is_minimal"] = False
    full_rows = df[df["is_full_dimension"].map(_as_bool)]

    for (_, _), group in full_rows.groupby(["h", "exponent"]):
        architectures = {
            index: _parse_architecture(row["architecture"])
            for index, row in group.iterrows()
        }
        for index, architecture in architectures.items():
            df.at[index, "is_minimal"] = not any(
                other_index != index
                and other_architecture != architecture
                and is_less_or_equal(other_architecture, architecture)
                for other_index, other_architecture in architectures.items()
            )
    return df


def evaluate_single_architecture(
    hidden_tuple: tuple,
    h: int,
    d_0: int,
    d_h: int,
    exponent: int,
):
    """Evaluate one architecture using the unchanged external compute_dimension."""
    sizes = [d_0] + list(hidden_tuple) + [d_h]
    arch_str = str(sizes)

    try:
        _, _, amb, _, dim, _ = compute_dimension(sizes, exponent)
        params = calculate_parameter_count(hidden_tuple, d_0, d_h)
        is_full = dim == amb

        status = "FULL " if is_full else "SHORT"
        print(
            f"  [{status}] {arch_str} -> Rank: {dim}/{amb} "
            f"(Params: {params})",
            flush=True,
        )

        return {
            "h": h,
            "exponent": exponent,
            "architecture": arch_str,
            "num_parameters": params,
            "dimension_computed": int(dim),
            "ambient_dimension": int(amb),
            "is_full_dimension": bool(is_full),
            "is_minimal": False,
        }
    except Exception as exc:
        print(f"  [ERROR] {arch_str} failed: {exc}", flush=True)
        return None

In [4]:
# GPU candidate generation and filtering

_GPU_GENERATE_AND_FILTER = None
_GPU_PRUNE_POOL = None


def _get_gpu_kernels():
    """Compile the fused CuPy kernels once, on first use."""
    global _GPU_GENERATE_AND_FILTER, _GPU_PRUNE_POOL

    if not _CUPY_AVAILABLE:
        raise RuntimeError("CuPy is not available.")

    if _GPU_GENERATE_AND_FILTER is None:
        _GPU_GENERATE_AND_FILTER = cp.RawKernel(
            r'''
            extern "C" __global__
            void generate_and_filter(
                const long long start,
                const long long count,
                const int num_hidden,
                const int* lower,
                const int* radices,
                const int d0,
                const int dh,
                const long long ambient_dim,
                const long long* evaluated_keys,
                const long long num_evaluated,
                const int* known_minimal,
                const int num_minimal,
                const int* known_short,
                const int num_short,
                int* candidates,
                unsigned char* keep,
                unsigned char* reason)
            {
                const long long tid =
                    (long long)blockDim.x * blockIdx.x + threadIdx.x;
                if (tid >= count) return;

                const long long key = start + tid;
                long long quotient = key;
                long long params = 0;
                int next_width = dh;

                for (int j = num_hidden - 1; j >= 0; --j) {
                    const int width = lower[j] +
                        (int)(quotient % (long long)radices[j]);
                    quotient /= (long long)radices[j];
                    candidates[tid * num_hidden + j] = width;
                    params += (long long)width * (long long)next_width;
                    next_width = width;
                }
                params += (long long)d0 *
                    (long long)candidates[tid * num_hidden];

                // Exact CSV reuse: binary search the sorted mixed-radix keys.
                long long left = 0;
                long long right = num_evaluated;
                while (left < right) {
                    const long long middle = left + (right - left) / 2;
                    if (evaluated_keys[middle] < key) left = middle + 1;
                    else right = middle;
                }
                if (left < num_evaluated && evaluated_keys[left] == key) {
                    keep[tid] = 0;
                    reason[tid] = 1;
                    return;
                }

                if (params < ambient_dim) {
                    keep[tid] = 0;
                    reason[tid] = 2;
                    return;
                }

                // Reject candidates above a known filling point.
                for (int m = 0; m < num_minimal; ++m) {
                    bool dominated = true;
                    for (int j = 0; j < num_hidden; ++j) {
                        if (known_minimal[m * num_hidden + j] >
                            candidates[tid * num_hidden + j]) {
                            dominated = false;
                            break;
                        }
                    }
                    if (dominated) {
                        keep[tid] = 0;
                        reason[tid] = 3;
                        return;
                    }
                }

                // Reject candidates below a known nonfilling point.
                for (int m = 0; m < num_short; ++m) {
                    bool dominated = true;
                    for (int j = 0; j < num_hidden; ++j) {
                        if (candidates[tid * num_hidden + j] >
                            known_short[m * num_hidden + j]) {
                            dominated = false;
                            break;
                        }
                    }
                    if (dominated) {
                        keep[tid] = 0;
                        reason[tid] = 4;
                        return;
                    }
                }

                keep[tid] = 1;
                reason[tid] = 0;
            }
            ''',
            "generate_and_filter",
        )

    if _GPU_PRUNE_POOL is None:
        _GPU_PRUNE_POOL = cp.RawKernel(
            r'''
            extern "C" __global__
            void prune_pool(
                const int* candidates,
                const long long count,
                const int num_hidden,
                const int* target,
                const int mode,
                unsigned char* keep)
            {
                const long long tid =
                    (long long)blockDim.x * blockIdx.x + threadIdx.x;
                if (tid >= count) return;

                bool remove = true;
                for (int j = 0; j < num_hidden; ++j) {
                    const int value = candidates[tid * num_hidden + j];
                    const int boundary = target[j];

                    // mode == 1: remove target <= candidate (upward orthant)
                    // mode == 0: remove candidate <= target (downward orthant)
                    if (mode == 1) {
                        if (boundary > value) {
                            remove = false;
                            break;
                        }
                    } else {
                        if (value > boundary) {
                            remove = false;
                            break;
                        }
                    }
                }
                keep[tid] = remove ? 0 : 1;
            }
            ''',
            "prune_pool",
        )

    return _GPU_GENERATE_AND_FILTER, _GPU_PRUNE_POOL


def _encode_hidden_in_box(
    hidden: Sequence[int],
    lower: Sequence[int],
    upper: Sequence[int],
) -> int | None:
    """Return the mixed-radix box index, or None when the point is outside."""
    key = 0
    for value, lo, hi in zip(hidden, lower, upper):
        if value < lo or value > hi:
            return None
        key = key * (hi - lo + 1) + (value - lo)
    return int(key)


def _cuda_available() -> bool:
    if not _CUPY_AVAILABLE:
        return False
    try:
        return cp.cuda.runtime.getDeviceCount() > 0
    except Exception:
        return False


def _choose_candidate_storage(
    formal_box_size: int,
    num_hidden: int,
    use_gpu: bool,
    gpu_storage_fraction: float,
) -> str:
    """Keep the live queue on GPU only when its conservative peak fits."""
    if not use_gpu:
        return "cpu"

    free_bytes, _ = cp.cuda.runtime.memGetInfo()
    candidate_bytes = formal_box_size * num_hidden * np.dtype(np.int32).itemsize
    permutation_bytes = formal_box_size * np.dtype(np.int64).itemsize
    conservative_peak = 2 * candidate_bytes + permutation_bytes

    return (
        "gpu"
        if conservative_peak <= int(free_bytes * gpu_storage_fraction)
        else "cpu"
    )


def _numpy_generate_and_filter_chunk(
    start: int,
    stop: int,
    *,
    lower: tuple[int, ...],
    upper: tuple[int, ...],
    d_0: int,
    d_h: int,
    ambient_dim: int,
    evaluated_keys: np.ndarray,
    known_minimal: tuple[tuple[int, ...], ...],
    known_short: tuple[tuple[int, ...], ...],
):
    """CPU fallback with the same mixed-radix generation and pruning rules."""
    num_hidden = len(lower)
    radices = np.asarray(
        [hi - lo + 1 for lo, hi in zip(lower, upper)],
        dtype=np.int64,
    )
    ids = np.arange(start, stop, dtype=np.int64)
    work = ids.copy()
    candidates = np.empty((stop - start, num_hidden), dtype=np.int32)

    for j in range(num_hidden - 1, -1, -1):
        candidates[:, j] = lower[j] + (work % radices[j]).astype(np.int32)
        work //= radices[j]

    active = np.ones(stop - start, dtype=bool)
    counts = np.zeros(5, dtype=np.int64)

    if evaluated_keys.size:
        positions = np.searchsorted(evaluated_keys, ids)
        exact = positions < evaluated_keys.size
        safe_positions = np.minimum(positions, evaluated_keys.size - 1)
        exact &= evaluated_keys[safe_positions] == ids
        counts[1] += np.count_nonzero(exact)
        active &= ~exact

    params = (
        d_0 * candidates[:, 0].astype(np.int64)
        + candidates[:, -1].astype(np.int64) * d_h
    )
    if num_hidden > 1:
        params += np.sum(
            candidates[:, :-1].astype(np.int64)
            * candidates[:, 1:].astype(np.int64),
            axis=1,
        )

    parameter_reject = active & (params < ambient_dim)
    counts[2] += np.count_nonzero(parameter_reject)
    active &= ~parameter_reject

    for boundary in known_minimal:
        reject = active & np.all(candidates >= np.asarray(boundary), axis=1)
        counts[3] += np.count_nonzero(reject)
        active &= ~reject

    for boundary in known_short:
        reject = active & np.all(candidates <= np.asarray(boundary), axis=1)
        counts[4] += np.count_nonzero(reject)
        active &= ~reject

    counts[0] = np.count_nonzero(active)
    return candidates[active], counts


def _build_candidate_pool(
    *,
    lower: tuple[int, ...],
    upper: tuple[int, ...],
    d_0: int,
    d_h: int,
    ambient_dim: int,
    evaluated_points: set[tuple[int, ...]],
    known_minimal: set[tuple[int, ...]],
    known_short: set[tuple[int, ...]],
    candidate_chunk_size: int,
    prefer_gpu: bool,
    gpu_storage_fraction: float,
):
    """
    Generate and filter the search box in chunks.

    On CUDA, one fused RawKernel performs mixed-radix candidate generation,
    exact-CSV rejection, parameter-count pruning, and both dominance tests.
    """
    num_hidden = len(lower)
    radices = tuple(hi - lo + 1 for lo, hi in zip(lower, upper))
    formal_box_size = math.prod(radices)

    use_gpu = bool(prefer_gpu and _cuda_available())
    storage = _choose_candidate_storage(
        formal_box_size,
        num_hidden,
        use_gpu,
        gpu_storage_fraction,
    )

    evaluated_keys = sorted(
        key
        for point in evaluated_points
        if (key := _encode_hidden_in_box(point, lower, upper)) is not None
    )
    minimal_tuple = tuple(sorted(known_minimal))
    short_tuple = tuple(sorted(known_short))

    candidate_chunks = []
    counts = np.zeros(5, dtype=np.int64)

    backend_label = "CuPy fused CUDA kernel" if use_gpu else "NumPy fallback"
    print(f"  Candidate backend: {backend_label}", flush=True)
    print(f"  Live queue storage: {storage.upper()}", flush=True)

    if use_gpu:
        generate_and_filter, _ = _get_gpu_kernels()
        lower_gpu = cp.asarray(lower, dtype=cp.int32)
        radices_gpu = cp.asarray(radices, dtype=cp.int32)
        evaluated_gpu = cp.asarray(evaluated_keys, dtype=cp.int64)
        minimal_gpu = cp.asarray(minimal_tuple, dtype=cp.int32).reshape(-1)
        short_gpu = cp.asarray(short_tuple, dtype=cp.int32).reshape(-1)

    progress = tqdm(
        total=formal_box_size,
        desc="  GPU generate/filter" if use_gpu else "  CPU generate/filter",
        leave=False,
        dynamic_ncols=True,
        unit="cand",
    )

    try:
        for start in range(0, formal_box_size, candidate_chunk_size):
            stop = min(start + candidate_chunk_size, formal_box_size)
            count = stop - start

            if use_gpu:
                candidates_gpu = cp.empty(
                    (count, num_hidden),
                    dtype=cp.int32,
                )
                keep_gpu = cp.empty(count, dtype=cp.uint8)
                reason_gpu = cp.empty(count, dtype=cp.uint8)

                threads = 256
                blocks = (count + threads - 1) // threads
                generate_and_filter(
                    (blocks,),
                    (threads,),
                    (
                        np.int64(start),
                        np.int64(count),
                        np.int32(num_hidden),
                        lower_gpu,
                        radices_gpu,
                        np.int32(d_0),
                        np.int32(d_h),
                        np.int64(ambient_dim),
                        evaluated_gpu,
                        np.int64(len(evaluated_keys)),
                        minimal_gpu,
                        np.int32(len(minimal_tuple)),
                        short_gpu,
                        np.int32(len(short_tuple)),
                        candidates_gpu,
                        keep_gpu,
                        reason_gpu,
                    ),
                )

                chunk_counts = cp.asnumpy(
                    cp.bincount(reason_gpu, minlength=5)
                ).astype(np.int64, copy=False)
                counts += chunk_counts

                survivors_gpu = candidates_gpu[keep_gpu.astype(cp.bool_)]
                if storage == "gpu":
                    candidate_chunks.append(survivors_gpu)
                else:
                    candidate_chunks.append(cp.asnumpy(survivors_gpu))
            else:
                survivors_cpu, chunk_counts = _numpy_generate_and_filter_chunk(
                    start,
                    stop,
                    lower=lower,
                    upper=upper,
                    d_0=d_0,
                    d_h=d_h,
                    ambient_dim=ambient_dim,
                    evaluated_keys=np.asarray(evaluated_keys, dtype=np.int64),
                    known_minimal=minimal_tuple,
                    known_short=short_tuple,
                )
                counts += chunk_counts
                candidate_chunks.append(survivors_cpu)

            progress.update(count)
    finally:
        progress.close()

    if candidate_chunks:
        if storage == "gpu":
            pool = cp.concatenate(candidate_chunks, axis=0)
        else:
            pool = np.concatenate(candidate_chunks, axis=0)
    else:
        pool = (
            cp.empty((0, num_hidden), dtype=cp.int32)
            if storage == "gpu"
            else np.empty((0, num_hidden), dtype=np.int32)
        )

    # Preserve the original randomized search order without Python tuples.
    if len(pool) > 1:
        if storage == "gpu":
            pool = pool[cp.random.permutation(len(pool))]
        else:
            np.random.default_rng().shuffle(pool, axis=0)

    stats = {
        "formal_box_size": int(formal_box_size),
        "survivors": int(counts[0]),
        "exact_csv": int(counts[1]),
        "parameter": int(counts[2]),
        "above_filling": int(counts[3]),
        "below_short": int(counts[4]),
        "backend": backend_label,
        "storage": storage,
    }
    return pool, stats


def _pool_pop(pool):
    """Pop from the end so no array shift is required."""
    if _CUPY_AVAILABLE and isinstance(pool, cp.ndarray):
        target = tuple(int(x) for x in cp.asnumpy(pool[-1]))
    else:
        target = tuple(int(x) for x in pool[-1])
    return target, pool[:-1]


def _prune_pool(pool, target: tuple[int, ...], *, upward: bool):
    """Vectorized queue compaction; CUDA uses a fused comparison kernel."""
    before = len(pool)
    if before == 0:
        return pool, 0

    if _CUPY_AVAILABLE and isinstance(pool, cp.ndarray):
        _, prune_kernel = _get_gpu_kernels()
        target_gpu = cp.asarray(target, dtype=cp.int32)
        keep_gpu = cp.empty(before, dtype=cp.uint8)
        threads = 256
        blocks = (before + threads - 1) // threads
        prune_kernel(
            (blocks,),
            (threads,),
            (
                pool,
                np.int64(before),
                np.int32(pool.shape[1]),
                target_gpu,
                np.int32(1 if upward else 0),
                keep_gpu,
            ),
        )
        pool = pool[keep_gpu.astype(cp.bool_)]
    else:
        boundary = np.asarray(target, dtype=np.int32)
        if upward:
            keep = ~np.all(pool >= boundary, axis=1)
        else:
            keep = ~np.all(pool <= boundary, axis=1)
        pool = pool[keep]

    return pool, before - len(pool)


In [5]:
# Core Search & Pruning Algorithm

def parameter_boundary_search(
    h_values: list,
    max_width=6,
    min_width=1,
    exponent=2,
    d_0=2,
    d_h=1,
    csv_filename="architecture_search_log.csv",
    user_guesses=None,
    layer_bounds=None,
    *,
    candidate_chunk_size=262_144,
    prefer_gpu_candidate_filter=True,
    gpu_storage_fraction=0.35,
    csv_checkpoint_every=1,
):
    """
    Randomized boundary search with GPU-accelerated candidate handling.

    The original public arguments and workflow are unchanged. Additional tuning
    arguments are keyword-only, so all existing calls continue to run as-is.

    Preserved features
    ------------------
    - the external ``compute_dimension`` implementation loaded above;
    - global or custom per-layer bounds;
    - user guesses;
    - randomized sequential architecture evaluation;
    - monotonic upward/downward pruning;
    - the original CSV schema and exact CSV path;
    - periodic CSV updates and safe KeyboardInterrupt handling.

    Candidate-side speedups
    -----------------------
    - mixed-radix generation avoids ``itertools.product`` and Python tuples;
    - a fused CuPy RawKernel generates and filters each chunk on the GPU;
    - prior evaluations are rejected by integer-key binary search;
    - parameter and dominance pruning occur inside the same GPU kernel;
    - the live queue is a compact tensor/array rather than a Python list;
    - queue pruning after each evaluation is a fused GPU mask when it fits.
    """
    if candidate_chunk_size <= 0:
        raise ValueError("candidate_chunk_size must be positive.")
    if not (0 < gpu_storage_fraction <= 1):
        raise ValueError("gpu_storage_fraction must lie in (0, 1].")
    if csv_checkpoint_every <= 0:
        raise ValueError("csv_checkpoint_every must be positive.")

    required_columns = [
        "h",
        "exponent",
        "architecture",
        "num_parameters",
        "dimension_computed",
        "ambient_dimension",
        "is_full_dimension",
        "is_minimal",
    ]
    csv_path = Path(csv_filename)

    if csv_path.exists():
        print(f"Loading existing database from '{csv_path}'...", flush=True)
        df = pd.read_csv(csv_path)
        for column in required_columns:
            if column not in df.columns:
                df[column] = (
                    False
                    if column in {"is_full_dimension", "is_minimal"}
                    else None
                )
        df = df.drop_duplicates(
            subset=["h", "exponent", "architecture"],
            keep="last",
        )
    else:
        print("No existing database found. Starting fresh...", flush=True)
        df = pd.DataFrame(columns=required_columns)

    completed_since_checkpoint = 0

    def checkpoint(*, recompute_minimality: bool = False) -> None:
        nonlocal df, completed_since_checkpoint
        if recompute_minimality:
            df = _recompute_minimality(df)
        _save_csv(df, csv_path)
        completed_since_checkpoint = 0

    try:
        for h in h_values:
            print(
                f"\n--- RANDOM SEARCH: h={h}, exponent={exponent} ---",
                flush=True,
            )

            num_hidden = h - 1
            if num_hidden <= 0:
                print(
                    f"  [WARNING] h={h} has no hidden layers. Skipping.",
                    flush=True,
                )
                continue

            degree = exponent ** num_hidden
            ambient_dim = math.comb(
                degree + d_0 - 1,
                d_0 - 1,
            ) * d_h
            print(f"  Target Ambient Dimension: {ambient_dim}", flush=True)

            evaluation_status: dict[tuple[int, ...], bool] = {}
            if not df.empty:
                subset_df = df[
                    (df["h"] == h)
                    & (df["exponent"] == exponent)
                ]
                for _, row in subset_df.iterrows():
                    architecture = _parse_architecture(row["architecture"])
                    if (
                        len(architecture) == h + 1
                        and architecture[0] == d_0
                        and architecture[-1] == d_h
                    ):
                        evaluation_status[architecture[1:-1]] = _as_bool(
                            row["is_full_dimension"]
                        )

            known_minimal = _minimal_antichain(
                hidden
                for hidden, is_full in evaluation_status.items()
                if is_full
            )
            known_short = _maximal_antichain(
                hidden
                for hidden, is_full in evaluation_status.items()
                if not is_full
            )

            print(
                f"  Loaded {len(evaluation_status):,} prior evaluation(s).",
                flush=True,
            )
            print(
                f"  Filling antichain size: {len(known_minimal):,}; "
                f"nonfilling antichain size: {len(known_short):,}.",
                flush=True,
            )

            # Evaluate user guesses before constructing the candidate pool.
            if user_guesses and h in user_guesses:
                for raw_guess in user_guesses[h]:
                    guess = tuple(int(x) for x in raw_guess)
                    if len(guess) != num_hidden:
                        print(
                            f"  [WARNING] Guess {guess} has length "
                            f"{len(guess)}, but h={h} requires "
                            f"{num_hidden}. Skipping it.",
                            flush=True,
                        )
                        continue
                    if guess in evaluation_status:
                        print(
                            f"  Guess {guess} already appears in the CSV: "
                            f"{'FILLING' if evaluation_status[guess] else 'SHORT'}.",
                            flush=True,
                        )
                        continue

                    print(f"  Evaluating Guess: {guess}...", flush=True)
                    result = evaluate_single_architecture(
                        guess,
                        h,
                        d_0,
                        d_h,
                        exponent,
                    )
                    if not result:
                        continue

                    is_full = _as_bool(result["is_full_dimension"])
                    evaluation_status[guess] = is_full
                    df = _append_result(
                        df,
                        result,
                        hidden=guess,
                        h=h,
                        exponent=exponent,
                        d_0=d_0,
                        d_h=d_h,
                        ambient_dim=ambient_dim,
                    )
                    completed_since_checkpoint += 1
                    checkpoint()

                    if is_full:
                        known_minimal = _insert_minimal(known_minimal, guess)
                    else:
                        known_short = _insert_maximal(known_short, guess)

            # Resolve exactly the same custom/global search box as before.
            if layer_bounds and h in layer_bounds:
                bounds = layer_bounds[h]
                if len(bounds) != num_hidden:
                    print(
                        f"  [WARNING] layer_bounds for h={h} has length "
                        f"{len(bounds)}, but expected {num_hidden}. "
                        "Skipping this depth...",
                        flush=True,
                    )
                    continue
                lower = tuple(max(1, int(lo)) for lo, _ in bounds)
                upper = tuple(int(hi) for _, hi in bounds)
                print(f"  Using custom per-layer bounds: {bounds}", flush=True)
            else:
                lo = max(1, int(min_width))
                hi = int(max_width)
                lower = (lo,) * num_hidden
                upper = (hi,) * num_hidden
                print(
                    f"  Using global bounds: min_width={min_width}, "
                    f"max_width={max_width}",
                    flush=True,
                )

            if any(lo > hi for lo, hi in zip(lower, upper)):
                print(
                    f"  [WARNING] Inconsistent bounds: lower={lower}, "
                    f"upper={upper}. Skipping h={h}.",
                    flush=True,
                )
                continue

            formal_box_size = math.prod(
                hi - lo + 1 for lo, hi in zip(lower, upper)
            )
            print(f"  Formal box size: {formal_box_size:,}", flush=True)
            print("  Generating and filtering candidates in chunks...", flush=True)

            candidate_pool, filter_stats = _build_candidate_pool(
                lower=lower,
                upper=upper,
                d_0=d_0,
                d_h=d_h,
                ambient_dim=ambient_dim,
                evaluated_points=set(evaluation_status),
                known_minimal=known_minimal,
                known_short=known_short,
                candidate_chunk_size=int(candidate_chunk_size),
                prefer_gpu=bool(prefer_gpu_candidate_filter),
                gpu_storage_fraction=float(gpu_storage_fraction),
            )

            print(
                f"  Exact CSV rejections:       {filter_stats['exact_csv']:,}",
                flush=True,
            )
            print(
                f"  Parameter-count rejections: {filter_stats['parameter']:,}",
                flush=True,
            )
            print(
                f"  Above-filling rejections:   "
                f"{filter_stats['above_filling']:,}",
                flush=True,
            )
            print(
                f"  Below-short rejections:     "
                f"{filter_stats['below_short']:,}",
                flush=True,
            )
            print(
                f"  Starting sequential evaluation on "
                f"{len(candidate_pool):,} viable candidates.",
                flush=True,
            )

            new_evaluations = 0
            dynamic_upward_prunes = 0
            dynamic_downward_prunes = 0

            while len(candidate_pool):
                target, candidate_pool = _pool_pop(candidate_pool)
                result = evaluate_single_architecture(
                    target,
                    h,
                    d_0,
                    d_h,
                    exponent,
                )
                if not result:
                    continue

                is_full = _as_bool(result["is_full_dimension"])
                evaluation_status[target] = is_full
                df = _append_result(
                    df,
                    result,
                    hidden=target,
                    h=h,
                    exponent=exponent,
                    d_0=d_0,
                    d_h=d_h,
                    ambient_dim=ambient_dim,
                )
                new_evaluations += 1
                completed_since_checkpoint += 1

                if completed_since_checkpoint >= csv_checkpoint_every:
                    checkpoint()

                if is_full:
                    known_minimal = _insert_minimal(known_minimal, target)
                    candidate_pool, pruned = _prune_pool(
                        candidate_pool,
                        target,
                        upward=True,
                    )
                    dynamic_upward_prunes += pruned
                    if pruned:
                        print(
                            f"    [Queue Upward Pruned] {pruned:,} "
                            "supersets removed from queue.",
                            flush=True,
                        )
                else:
                    known_short = _insert_maximal(known_short, target)
                    candidate_pool, pruned = _prune_pool(
                        candidate_pool,
                        target,
                        upward=False,
                    )
                    dynamic_downward_prunes += pruned
                    if pruned:
                        print(
                            f"    [Queue Downward Pruned] {pruned:,} "
                            "subsets removed from queue.",
                            flush=True,
                        )

            print(f"\n  Search complete for h={h}.", flush=True)
            print(
                f"  New architecture evaluations: {new_evaluations:,}",
                flush=True,
            )
            print(
                f"  Dynamic upward queue prunes:  "
                f"{dynamic_upward_prunes:,}",
                flush=True,
            )
            print(
                f"  Dynamic downward queue prunes: "
                f"{dynamic_downward_prunes:,}",
                flush=True,
            )

            checkpoint(recompute_minimality=True)
            print(
                f"Database successfully saved to '{csv_path}'.",
                flush=True,
            )

    except KeyboardInterrupt:
        print(
            "\n[Interrupt] User halted the search. Saving all completed "
            "evaluations to the same CSV...",
            flush=True,
        )
        checkpoint(recompute_minimality=True)
        print(
            f"Database successfully saved to '{csv_path}'.",
            flush=True,
        )
        print(
            "Run the same cell again to reuse the CSV and continue pruning.",
            flush=True,
        )
        return df

    checkpoint(recompute_minimality=True)
    print(
        f"\nAll requested searches completed. Database saved to "
        f"'{csv_path}'.",
        flush=True,
    )
    return df


# Execute Search

In [ ]:
# Looped version for searches

my_guesses = globals().get("my_guesses", {})

for d0 in range(3,5):
    for dh in range(1,3):
        for r in range(2,4):
            h_values_to_test = [2,3,4,5,6,7] 

            def bound(L, i, r, d0, dh):
                return (1, min(dh * r**(i*d0), math.comb(r**(L-i) + d0 - 1, r**(L-i))))
            
            def double_bound(L, i, r, d0, dh):
                return (1, 2*min(dh * r**(i*d0), math.comb(r**(L-i) + d0 - 1, r**(L-i))))

            custom_bounds = {
                L: [bound(L, i, r, d0, dh) for i in range(1, L)]
                for L in range(1, 15)  # depths 1 through 8
            }


            df_results = parameter_boundary_search(
                h_values=h_values_to_test, 
                max_width=100,        # Fallback if `h` not in layer_bounds
                min_width=1,         # Fallback if `h` not in layer_bounds
                exponent=r,          
                d_0=d0,              
                d_h=dh,              
                csv_filename=f"../data/raw/{d0}_{dh}_r{r}_architectures.csv",
                user_guesses=my_guesses,
                layer_bounds=custom_bounds,  # Injecting the new feature here
                candidate_chunk_size=262_144,
                prefer_gpu_candidate_filter=True,
                gpu_storage_fraction=0.50,
                csv_checkpoint_every=1,
            )

Loading existing database from '..\data\raw\3_1_r2_architectures.csv'...

--- RANDOM SEARCH: h=2, exponent=2 ---
  Target Ambient Dimension: 6
  Loaded 6 prior evaluation(s).
  Filling antichain size: 1; nonfilling antichain size: 1.
  Using custom per-layer bounds: [(1, 6)]
  Formal box size: 6
  Generating and filtering candidates in chunks...
  Candidate backend: CuPy fused CUDA kernel


C:\Users\daoke\AppData\Local\Temp\ipykernel_3132\660216561.py:65: DtypeWarning: Columns (0: shrunk_hidden_layers) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


  Live queue storage: GPU


  Exact CSV rejections:       5
  Parameter-count rejections: 0
  Above-filling rejections:   1
  Below-short rejections:     0
  Starting sequential evaluation on 0 viable candidates.

  Search complete for h=2.
  New architecture evaluations: 0
  Dynamic upward queue prunes:  0
  Dynamic downward queue prunes: 0


Database successfully saved to '..\data\raw\3_1_r2_architectures.csv'.

--- RANDOM SEARCH: h=3, exponent=2 ---
  Target Ambient Dimension: 15
  Loaded 101 prior evaluation(s).
  Filling antichain size: 1; nonfilling antichain size: 2.
  Using custom per-layer bounds: [(1, 8), (1, 6)]
  Formal box size: 48
  Generating and filtering candidates in chunks...
  Candidate backend: CuPy fused CUDA kernel
  Live queue storage: GPU


...

  Exact CSV rejections:       44,816
  Parameter-count rejections: 567,037
  Above-filling rejections:   71,263
  Below-short rejections:     261,350
  Starting sequential evaluation on 1,129,134 viable candidates.
  [SHORT] [3, 8, 14, 25, 5, 4, 1] -> Rank: 234/561 (Params: 635)
    [Queue Downward Pruned] 68 subsets removed from queue.
  [SHORT] [3, 8, 48, 17, 1, 5, 1] -> Rank: 45/561 (Params: 1251)
    [Queue Downward Pruned] 6,486 subsets removed from queue.
  [SHORT] [3, 2, 58, 33, 12, 2, 1] -> Rank: 21/561 (Params: 2458)
    [Queue Downward Pruned] 29,657 subsets removed from queue.
  [SHORT] [3, 7, 33, 25, 5, 2, 1] -> Rank: 229/561 (Params: 1214)
    [Queue Downward Pruned] 4,271 subsets removed from queue.
  [SHORT] [3, 3, 2, 44, 10, 2, 1] -> Rank: 19/561 (Params: 565)
  [SHORT] [3, 6, 35, 9, 9, 3, 1] -> Rank: 222/561 (Params: 654)
    [Queue Downward Pruned] 150 subsets removed from queue.
  [SHORT] [3, 1, 40, 36, 3, 1, 1] -> Rank: 3/561 (Params: 1595)
    [Queue Downward Prune

  Exact CSV rejections:       145
  Parameter-count rejections: 85,697,168
  Above-filling rejections:   1,465,117
  Below-short rejections:     3,449,490
  Starting sequential evaluation on 226,648,880 viable candidates.
  [SHORT] [3, 1, 13, 52, 24, 11, 6, 1] -> Rank: 3/2145 (Params: 2276)
    [Queue Downward Pruned] 91 subsets removed from queue.
  [SHORT] [3, 2, 41, 97, 35, 8, 3, 1] -> Rank: 37/2145 (Params: 7767)
    [Queue Downward Pruned] 2,663,629 subsets removed from queue.
  [SHORT] [3, 4, 59, 27, 37, 2, 2, 1] -> Rank: 269/2145 (Params: 2920)
    [Queue Downward Pruned] 14,916 subsets removed from queue.
  [SHORT] [3, 4, 53, 74, 19, 3, 2, 1] -> Rank: 400/2145 (Params: 5617)
    [Queue Downward Pruned] 329,663 subsets removed from queue.
  [SHORT] [3, 1, 46, 143, 38, 5, 5, 1] -> Rank: 3/2145 (Params: 12281)
    [Queue Downward Pruned] 2,854,831 subsets removed from queue.
  [SHORT] [3, 2, 44, 130, 17, 7, 1, 1] -> Rank: 21/2145 (Params: 8151)
    [Queue Downward Pruned] 198,083 

  Exact CSV rejections:       3
  Parameter-count rejections: 2
  Above-filling rejections:   5
  Below-short rejections:     0
  Starting sequential evaluation on 0 viable candidates.

  Search complete for h=2.
  New architecture evaluations: 0
  Dynamic upward queue prunes:  0
  Dynamic downward queue prunes: 0


Database successfully saved to '..\data\raw\3_1_r3_architectures.csv'.

--- RANDOM SEARCH: h=3, exponent=3 ---
  Target Ambient Dimension: 55
  Loaded 68 prior evaluation(s).
  Filling antichain size: 2; nonfilling antichain size: 3.
  Using custom per-layer bounds: [(1, 27), (1, 10)]
  Formal box size: 270
  Generating and filtering candidates in chunks...
  Candidate backend: CuPy fused CUDA kernel
  Live queue storage: GPU


  Exact CSV rejections:       17
  Parameter-count rejections: 63
  Above-filling rejections:   95
  Below-short rejections:     95
  Starting sequential evaluation on 0 viable candidates.

  Search complete for h=3.
  New architecture evaluations: 0
  Dynamic upward queue prunes:  0
  Dynamic downward queue prunes: 0
